In [1]:
import os
import uuid
import json
import time
import random
from collections import defaultdict

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

In [2]:
BASE_DIR = "wiki_en_1000"
os.makedirs(BASE_DIR, exist_ok=True)


In [3]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []

    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
EMBEDDING_DIM = 384


In [5]:
client = QdrantClient(":memory:")

COLLECTION_NAME = "wiki_hf_1000"

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE
    )
)


C:\Users\shiva\AppData\Local\Temp\ipykernel_12616\3617452003.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [6]:
# # Delete Collection
# client.delete_collection(collection_name="wiki_docs")

In [27]:
from qdrant_client.models import PointStruct

points = []
point_id = 0

for filename in os.listdir(BASE_DIR):
    file_path = os.path.join(BASE_DIR, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = chunk_text(text)
    chunks = chunks[:MAX_CHUNKS_PER_DOC]

    for i in range(0, len(chunks), EMBED_BATCH_SIZE):
        batch_chunks = chunks[i:i + EMBED_BATCH_SIZE]
        embeddings = model.encode(batch_chunks).tolist()

        for chunk, emb in zip(batch_chunks, embeddings):
            points.append(
                PointStruct(
                    id=point_id,
                    vector=emb,
                    payload={
                        "article_id": filename.replace(".txt", ""),
                        "text": chunk
                    }
                )
            )
            point_id += 1


In [28]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
query = "What is machine learning?"
query_embedding = model.encode(query).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=5,
    with_payload=True
).points


In [30]:
for i, hit in enumerate(results):
    print(f"\nResult {i+1}")
    print("Article ID:", hit.payload["article_id"])



Result 1
Article ID: 50447852

Result 2
Article ID: 48892446

Result 3
Article ID: 48892446

Result 4
Article ID: 1875925

Result 5
Article ID: 2073462


# Queriying

In [31]:
with open("queries_1000.json", "r", encoding="utf-8") as f:
    loaded_queries = json.load(f)

print(len(loaded_queries))
print(loaded_queries[:5])


1000
['when is the last episode of season 8 of the walking dead', 'in greek mythology who was the goddess of spring growth', 'benefits of colonial life for single celled organisms', 'how many season of the man in the high castle', 'who was the first ministry head of state in nigeria']


In [32]:
query_embeddings = model.encode(
    loaded_queries,
    batch_size=8,
    show_progress_bar=True
).tolist()


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

In [33]:
all_results = []

for emb in query_embeddings:
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=emb,
        limit=3,
        with_payload=True
    ).points
    all_results.append(res)


In [34]:
import time
import random
from collections import defaultdict

def run_queries_metrics(
    client,
    collection_name,
    model,
    queries,
    k=3,
    batch_size=8,
    num_examples=5
):
    per_query_results = defaultdict(list)
    query_latencies = []

    total_start = time.perf_counter()

    query_embeddings = model.encode(
        queries,
        batch_size=batch_size,
        show_progress_bar=True
    ).tolist()

    for q, emb in zip(queries, query_embeddings):
        start = time.perf_counter()

        results = client.query_points(
            collection_name=collection_name,
            query=emb,
            limit=k,
            with_payload=True
        ).points

        latency = time.perf_counter() - start
        query_latencies.append(latency)

        for rank, hit in enumerate(results, start=1):
            per_query_results[q].append({
                "rank": rank,
                "article_id": hit.payload["article_id"],
                "text": hit.payload["text"]
            })

    total_time = time.perf_counter() - total_start
    avg_latency = sum(query_latencies) / len(query_latencies)
    throughput = len(queries) / total_time


    print("\n=== QUERY EVALUATION SUMMARY ===")
    print(f"Total queries        : {len(queries)}")
    print(f"Total time (s)       : {total_time:.2f}")
    print(f"Avg latency (s)      : {avg_latency:.4f}")
    print(f"Throughput (q/s)     : {throughput:.2f}")

    print("\n=== SAMPLE QUERY RESULTS ===")

    example_queries = random.sample(
        queries, min(num_examples, len(queries))
    )

    for idx, q in enumerate(example_queries, 1):
        print(f"\nExample {idx}")
        print(f"Query: {q}")

        for res in per_query_results[q]:
            print(f"  → Rank {res['rank']}")
            print(f"    Article ID : {res['article_id']}")
            print(f"    Text       : {res['text'][:150]}...")
    

    return {
        "queries": len(queries),
        "total_time": total_time,
        "avg_latency": avg_latency,
        "throughput": throughput,
        "results": per_query_results
    }


In [36]:
with open("queries_1000.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

metrics = run_queries_metrics(
    client=client,
    collection_name=COLLECTION_NAME,
    model=model,
    queries=queries,
    k=3,
    batch_size=8,
    num_examples=5
)

Batches:   0%|          | 0/125 [00:00<?, ?it/s]


=== QUERY EVALUATION SUMMARY ===
Total queries        : 1000
Total time (s)       : 21.72
Avg latency (s)      : 0.0106
Throughput (q/s)     : 46.03

=== SAMPLE QUERY RESULTS ===

Example 1
Query: who is the current director of the national park service
  → Rank 1
    Article ID : 19234751
    Text       : TITLE: Ministry of Land and Resources of the People's Republic of China The Ministry of Land and Resources (MLR) of the People's Republic of China is ...
  → Rank 2
    Article ID : 3745212
    Text       : Democratic Institutions, 2017–present Herb Gray (BCom 1952) – Deputy Prime Minister of Canada, 1997–2002 Don Johnston (BA 1955, BCL 1958) – Minister o...
  → Rank 3
    Article ID : 48202044
    Text       : TITLE: James Cullingham James Cullingham Ph.D (born March 5, 1954) is a documentary filmmaker, historian and journalist with Tamarack Productions base...

Example 2
Query: how much do you win on family feud
  → Rank 1
    Article ID : 25986662
    Text       : in the 18-49 Ni